# Banking BERT Lab — запуск в Colab

Ноутбук для реального дообучения небольшой BERT-модели на BANKING77. **Сначала опубликуй проект в `Natalie897/banking-bert-lab`.** Если он ещё не опубликован, клонирование не сработает. В меню `Runtime → Change runtime type` можно выбрать GPU. До завершения запуска метрики не известны.

In [ ]:
%cd /content
!test -d banking-bert-lab || git clone https://github.com/Natalie897/banking-bert-lab.git
%cd /content/banking-bert-lab


## Установка зависимостей

При первом запуске загружаются Python-библиотеки, затем предобученная BERT и открытый BANKING77. Интернет обязателен.

In [ ]:
!pip -q install -r requirements.txt -r requirements-ci.txt

## Проверка тестов

Тесты проверяют подготовку данных и настоящий PyTorch train loop на простой локальной модели; они **не являются подтверждением качества BERT на BANKING77**.

In [ ]:
!python -m pytest -q

## Проверь доступный ускоритель

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Обучение и сравнение

Используем 8 классов, 70 примеров на класс, отдельные validation/test. Скрипт выводит макро-F1 после каждой эпохи. На CPU можно поставить `--epochs 1 --train-per-class 20` для очень короткой проверки; это не полноценная оценка.

In [ ]:
!python -m banking_bert_lab train --epochs 3 --train-per-class 70 --test-per-class 25

## Результаты

Это фактически вычисленные метрики. Сравниваем BERT и baseline на **одном** test. Результат зависит от запуска — готовые числа не подставлены.

In [ ]:
import json
from pathlib import Path
p = Path("artifacts/banking_bert/metrics.json")
r = json.loads(p.read_text(encoding="utf-8"))
print("Validation BERT macro-F1:", r["bert"]["validation"]["macro_f1"])
print("Test BERT macro-F1:", r["bert"]["test"]["macro_f1"])
print("Test TF-IDF macro-F1:", r["baseline"]["test"]["macro_f1"])
print("Best epoch:", r["bert"]["best_epoch"])

## Пример предсказания

Если score низкий, `review_required` ставится в `true`. Он означает необходимость проверки, а не гарантированную ошибку.

In [ ]:
!python -m banking_bert_lab predict --text "My bank card has not arrived yet"

## Ошибки модели

In [ ]:
import pandas as pd
errors = pd.read_csv("artifacts/banking_bert/bert_errors.csv")
display(errors.head(10))

## Скачай метрики

Можно сохранить `metrics.json` и собственные выводы для README, не публикуя веса модели. Результаты воспроизводимы при фиксированном seed с возможными небольшими расхождениями между CPU/GPU.

In [ ]:
from google.colab import files
files.download("artifacts/banking_bert/metrics.json")